In [1]:
import pandas as pd
import pickle
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

In [2]:
# Load dataset
df = pd.read_parquet("../data/raw/green_tripdata_2023-01.parquet")

In [3]:
print('First 5 rows:')
df.head()

First 5 rows:


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,1.0,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75
1,2,2023-01-01 00:51:03,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,1.0,0.5,2.64,0.0,None,1.0,15.84,1.0,1.0,0.00
2,2,2023-01-01 00:35:12,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,1.0,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00
3,1,2023-01-01 00:13:14,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,0.5,1.5,1.70,0.0,None,1.0,10.20,1.0,1.0,0.00
4,1,2023-01-01 00:33:04,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,0.5,1.5,0.00,0.0,None,1.0,8.00,1.0,1.0,0.00


In [4]:
print(f"Dataset shape: {df.shape}")


Dataset shape: (68211, 20)


In [5]:
print('Dataset Info:')
df.info()

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68211 entries, 0 to 68210
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               68211 non-null  int64         
 1   lpep_pickup_datetime   68211 non-null  datetime64[us]
 2   lpep_dropoff_datetime  68211 non-null  datetime64[us]
 3   store_and_fwd_flag     63887 non-null  object        
 4   RatecodeID             63887 non-null  float64       
 5   PULocationID           68211 non-null  int64         
 6   DOLocationID           68211 non-null  int64         
 7   passenger_count        63887 non-null  float64       
 8   trip_distance          68211 non-null  float64       
 9   fare_amount            68211 non-null  float64       
 10  extra                  68211 non-null  float64       
 11  mta_tax                68211 non-null  float64       
 12  tip_amount             68211 non-null  float64

In [6]:
# Sum Null values in every feature
df.isna().sum()

VendorID                     0
lpep_pickup_datetime         0
lpep_dropoff_datetime        0
store_and_fwd_flag        4324
RatecodeID                4324
PULocationID                 0
DOLocationID                 0
passenger_count           4324
trip_distance                0
fare_amount                  0
extra                        0
mta_tax                      0
tip_amount                   0
tolls_amount                 0
ehail_fee                68211
improvement_surcharge        0
total_amount                 0
payment_type              4324
trip_type                 4334
congestion_surcharge      4324
dtype: int64

In [7]:
# Calculate the target feature "Duration in minutes"
df["duration_min"] = (df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]).dt.total_seconds() / 60

In [8]:
# Display statistics for "duration_min" feature
df["duration_min"].describe()

count    68211.000000
mean        18.097271
std         74.925631
min          0.000000
25%          7.216667
50%         11.400000
75%         17.483333
max       1439.883333
Name: duration_min, dtype: float64

In [9]:
# Remove trips with outliers values under 1 min and above 60 mins
df = df[df["duration_min"].between(1, 60).copy()]
df["duration_min"].describe()

count    65946.000000
mean        13.747575
std          9.023810
min          1.000000
25%          7.483333
50%         11.533333
75%         17.450000
max         60.000000
Name: duration_min, dtype: float64

In [10]:
# Combine pickup and dropoff zones in one feature
df["PU_DO"] = df["PULocationID"].astype(str) + "_" + df["DOLocationID"].astype(str)


In [11]:
# Extract input features and target
input_features = df[["PU_DO", "trip_distance"]]
target = df["duration_min"]

In [12]:
# Splitting dataset
train_features, test_features, train_targets, test_targets = train_test_split(input_features, target, test_size=0.2, random_state=42)

In [13]:
# Convert feature dataframes into dictionaries
train_dict = train_features.to_dict(orient="records")
test_dict = test_features.to_dict(orient="records")

In [14]:
# Define the pipeline, fit the data and make predictions
pipeline = Pipeline([
    ("dv", DictVectorizer()),
    ("model", LinearRegression())
])

pipeline.fit(train_dict, train_targets)

predictions = pipeline.predict(test_dict)

In [15]:
# Evaluation
rmse = root_mean_squared_error(test_targets, predictions)
mae = mean_absolute_error(test_targets, predictions)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")

RMSE: 5.89
MAE: 3.86


In [16]:
# Save the pipeline
with open("../models/baseline.pkl", "wb") as f:
    pickle.dump(pipeline, f)